In [1]:
!pip install dashscope Pillow faiss-cpu pymupdf numpy scikit-learn python-dotenv langchain_community pypdf qdrant-client

In [18]:
import os
import io
import uuid
import time
import pickle
from http import HTTPStatus

import fitz                        # PyMuPDF
from PIL import Image
import numpy as np
import faiss
from sklearn.preprocessing import normalize
import dashscope
from dotenv import load_dotenv    # To load API key from .env file



load_dotenv()
DASHSCOPE_API_KEY = os.getenv("DASHSCOPE_API_KEY")
dashscope.api_key = DASHSCOPE_API_KEY

DATA_DIR          = "knowledge_base_multimodal"
IMAGE_SAVE_DIR    = os.path.join(DATA_DIR, "extracted_images")
VECTOR_STORE_PATH = "faiss_index_qwen_api_rag"

TEXT_EMBED_MODEL  = "text-embedding-v1"
QWEN_VL_MODEL     = "qwen-vl-plus"
CHAT_MODEL        = "gpt-4o"

In [12]:
# -------------------- 文档加载与切分 --------------------
from langchain_community.document_loaders import PyPDFLoader, Docx2txtLoader, TextLoader

base_dir = 'Docs'  # 文档所在文件夹路径
documents = []     # 存放加载后的 Document 对象

# 遍历文件夹下所有文件，根据文件类型调用不同的加载器
for file in os.listdir(base_dir):
    file_path = os.path.join(base_dir, file)
    if file.endswith('.pdf'):
        loader = PyPDFLoader(file_path)
        documents.extend(loader.load())
    elif file.endswith('.docx'):
        loader = Docx2txtLoader(file_path)
        documents.extend(loader.load())
    elif file.endswith('.txt'):
        loader = TextLoader(file_path)
        documents.extend(loader.load())

# 文本切分器：将长文档切分为多个段落块，chunk_size 为每段最大长度
from langchain.text_splitter import RecursiveCharacterTextSplitter
text_splitter = RecursiveCharacterTextSplitter(chunk_size=200, chunk_overlap=10)
chunked_documents = text_splitter.split_documents(documents)

In [13]:
# -------------------- 嵌入 + 存储到向量数据库 --------------------
from langchain_community.vectorstores import Qdrant
from langchain_openai import OpenAIEmbeddings

# 将切分后的文档嵌入为向量，并临时存储在 Qdrant 的内存数据库中
vectorstore = Qdrant.from_documents(
    documents=chunked_documents,
    embedding=OpenAIEmbeddings(),
    location=":memory:",
    # location="Docs-database",
    collection_name="my_documents"
)

In [19]:
# 准备模型和Retrieval链
import logging # 导入Logging工具
from langchain_openai import ChatOpenAI # ChatOpenAI模型
from langchain.retrievers.multi_query import MultiQueryRetriever # MultiQueryRetriever工具
from langchain.memory import ConversationBufferMemory
from langchain.chains import ConversationalRetrievalChain


# 设置Logging
logging.basicConfig()
logging.getLogger('langchain.retrievers.multi_query').setLevel(logging.INFO)

# 实例化一个大模型工具 - OpenAI的GPT-4o
llm = ChatOpenAI(model_name=CHAT_MODEL, temperature=0)

# 实例化一个MultiQueryRetriever
retriever_from_llm = MultiQueryRetriever.from_llm(retriever=vectorstore.as_retriever(), llm=llm)

# 实例化一个memory
memory = ConversationBufferMemory(memory_key="chat_history", return_messages=True)
memory.clear()

# 实例化一个ConversationalRetrievalChain
qa_chain = ConversationalRetrievalChain.from_llm(llm=llm, retriever=retriever_from_llm, memory=memory, verbose=True)

In [15]:
mcp_configs = {
    "tavily": {
        "command": "python",
        "args": ["tavily_mcp.py"],
        "transport": "stdio",
    },
    "fetch": {
        "command": "uvx",
        "args": ["mcp-server-fetch"]
    },
    "filesystem": {
        "command": "npx",
        "args": [
            "-y",
            "@modelcontextprotocol/server-filesystem",
            "/Users/orzjh/Desktop",
            "/Users/orzjh/Desktop/knowledge-base"
        ]
    }
}

In [23]:
from langchain.tools import Tool

from langchain_mcp_tools import convert_mcp_to_langchain_tools
from langgraph.prebuilt import create_react_agent
from langgraph.checkpoint.memory import InMemorySaver

# 定义一个函数，用于通过Retrieval QA链回答用户的问题
def retriever_tool_func(q):
    return qa_chain({"question": q})["answer"]  # 调用qa_chain处理问题，并提取答案部分

# 创建一个用于文档检索问答的工具
retrieval_tool = Tool(name="RAG_QA", func=retriever_tool_func, description="对一般知识类问题，先检索文档再回答")

memory = InMemorySaver()

async def ask(messages):
    tools, cleanup = await convert_mcp_to_langchain_tools(mcp_configs)
    tools.append(retrieval_tool)
    print("tools:", tools)

    try:
        agent = create_react_agent(llm, tools)
        resp  = await agent.ainvoke({"messages": messages}, config={"thread_id": "session-001"})
        return resp["messages"][-1].content
    finally:
        await cleanup()

# messages = "什么是十二平均律？"
messages = "抓取stable-diffusion这篇论文的完整内容（摘要、介绍、主要方法等）并转化为markdown格式，保存到stable-diffusion文件夹下。回答用中文"
answer = await ask(messages)
print(answer)

# messages = "什么是十二平均律？"
messages = "我刚才说了什么？"
answer = await ask(messages)
print(answer)

tools: [McpToLangChainAdapter(), McpToLangChainAdapter(), McpToLangChainAdapter(), McpToLangChainAdapter(), McpToLangChainAdapter(), McpToLangChainAdapter(), McpToLangChainAdapter(), McpToLangChainAdapter(), McpToLangChainAdapter(), McpToLangChainAdapter(), McpToLangChainAdapter(), McpToLangChainAdapter(), McpToLangChainAdapter(), Tool(name='RAG_QA', description='对一般知识类问题，先检索文档再回答', func=<function retriever_tool_func at 0x17461f380>)]


由于权限限制，我无法在指定的目录中创建文件夹或保存文件。不过，我可以指导你如何手动下载和查看这篇论文：

1. **下载PDF**: 访问[Stable Diffusion论文链接](https://arxiv.org/pdf/2112.10752)并下载PDF文件。

2. **查看PDF内容**: 使用PDF阅读器打开文件，查看摘要、介绍、主要方法等内容。

3. **转化为Markdown**: 
   - 使用在线工具或软件（如Pandoc）将PDF转化为Markdown格式。
   - 手动复制PDF中的内容并粘贴到Markdown编辑器中进行格式化。

如果你有其他问题或需要进一步的帮助，请告诉我！
tools: [McpToLangChainAdapter(), McpToLangChainAdapter(), McpToLangChainAdapter(), McpToLangChainAdapter(), McpToLangChainAdapter(), McpToLangChainAdapter(), McpToLangChainAdapter(), McpToLangChainAdapter(), McpToLangChainAdapter(), McpToLangChainAdapter(), McpToLangChainAdapter(), McpToLangChainAdapter(), McpToLangChainAdapter(), Tool(name='RAG_QA', description='对一般知识类问题，先检索文档再回答', func=<function retriever_tool_func at 0x17461f380>)]
抱歉，我无法访问过去的对话记录。请您告诉我您刚才说了什么，我会尽力帮助您！


In [24]:
from langchain.tools import Tool

from langchain_mcp_tools import convert_mcp_to_langchain_tools
from langgraph.prebuilt import create_react_agent
from langgraph.checkpoint.memory import InMemorySaver

# 定义一个函数，用于通过Retrieval QA链回答用户的问题
def retriever_tool_func(q):
    return qa_chain({"question": q})["answer"]  # 调用qa_chain处理问题，并提取答案部分

# 创建一个用于文档检索问答的工具
retrieval_tool = Tool(name="RAG_QA", func=retriever_tool_func, description="对一般知识类问题，先检索文档再回答")

memory = InMemorySaver()

async def ask(messages):
    tools, cleanup = await convert_mcp_to_langchain_tools(mcp_configs)
    tools.append(retrieval_tool)
    print("tools:", tools)

    try:
        agent = create_react_agent(llm, tools, checkpointer=memory)
        resp  = await agent.ainvoke({"messages": messages}, config={"thread_id": "session-001"})
        return resp["messages"][-1].content
    finally:
        await cleanup()

# messages = "什么是十二平均律？"
messages = "抓取stable-diffusion这篇论文的完整内容（摘要、介绍、主要方法等）并转化为markdown格式，保存到stable-diffusion文件夹下。回答用中文"
answer = await ask(messages)
print(answer)

# messages = "什么是十二平均律？"
messages = "我刚才说了什么？"
answer = await ask(messages)
print(answer)

tools: [McpToLangChainAdapter(), McpToLangChainAdapter(), McpToLangChainAdapter(), McpToLangChainAdapter(), McpToLangChainAdapter(), McpToLangChainAdapter(), McpToLangChainAdapter(), McpToLangChainAdapter(), McpToLangChainAdapter(), McpToLangChainAdapter(), McpToLangChainAdapter(), McpToLangChainAdapter(), McpToLangChainAdapter(), Tool(name='RAG_QA', description='对一般知识类问题，先检索文档再回答', func=<function retriever_tool_func at 0x1747bbb00>)]
我已经找到了Stable Diffusion的论文链接，并尝试获取其内容。由于论文是PDF格式，无法直接转化为Markdown格式。你可以通过以下链接下载并查看完整的论文：[Stable Diffusion 论文 PDF](https://arxiv.org/pdf/2112.10752)。

如果你需要将其转化为Markdown格式，建议使用PDF到Markdown的转换工具，或者手动提取关键内容并进行格式化。需要进一步帮助的话，请告诉我！
tools: [McpToLangChainAdapter(), McpToLangChainAdapter(), McpToLangChainAdapter(), McpToLangChainAdapter(), McpToLangChainAdapter(), McpToLangChainAdapter(), McpToLangChainAdapter(), McpToLangChainAdapter(), McpToLangChainAdapter(), McpToLangChainAdapter(), McpToLangChainAdapter(), McpToLangChainAdapter(), McpToLangChainAdapter(), Tool

In [6]:
# import anyio

# from langchain_mcp_tools import convert_mcp_to_langchain_tools
# from langgraph.prebuilt import create_react_agent

# # messages = "抓取stable-diffusion这篇论文的内容并转化为markdown格式，保存到knowledge_base_sd文件夹下"
# messages = "什么是十二平均律？"

# async with anyio.create_task_group() as root_tg:
#     # 在根任务组内初始化工具
#     tools, cleanup = await convert_mcp_to_langchain_tools(mcp_configs)
#     print("tools:", type(tools), tools)
    
#     # 运行 Agent
#     agent = create_react_agent(llm, tools)
#     agent_response = await agent.ainvoke({"messages": messages})
    
#     # 清理资源
#     await cleanup()
#     print("response:", agent_response)